# UMAP 3D — supervised vs unsupervised, OpenAI & TF-IDF

Two-step pipeline:

1. **Classify chatbot_arena queries via the CRAG label space.**
   Train a `LogisticRegression` (multiclass) on CRAG embeddings → CRAG `domain` ∈ {`finance`, `movie`, `music`, `sports`, `open`}, then predict labels for the 3000 chatbot_arena queries. One classifier per embedding type (OpenAI 1536-dim, TF-IDF 6032-dim).

   > Note: the user asked for *Linear Regression* — for multiclass classification the standard linear model is `LogisticRegression`. Same family (linear decision boundary on the embeddings).

2. **Visualize each embedding with UMAP in 3D, twice:**
   - **unsupervised** — UMAP ignores labels (geometry only).
   - **supervised** — `target_weight=0.01` so UMAP nudges the layout toward the labels but the geometric structure dominates.

Total: 4 UMAP plots (OpenAI unsup / OpenAI sup / TF-IDF unsup / TF-IDF sup). Same outlier-drop + centering pipeline as `umap_embeddings_3d.ipynb`.

Install deps if missing:
```
python -m pip install umap-learn plotly scikit-learn pandas
```

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

## 0. Load all 4 files

In [22]:
DATA_DIR = Path('/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed')

FILES = {
    # 'crag_openai': (DATA_DIR / 'crag_domain_open_ai_embeddings_subclass.json',           'open_ai_embeddings'),
    'crag_openai': (DATA_DIR / 'crag_domain_open_ai_embeddings.json',           'open_ai_embeddings'),
    'crag_tfidf':  (DATA_DIR / 'crag_tfidf_embeddings.json',                    'tf_idf_embedding'),
    'chat_openai': (DATA_DIR / 'chatbot_arena_open_ai_embeddings_sample3000.json', 'open_ai_embeddings'),
    'chat_tfidf':  (DATA_DIR / 'chatbot_arena_tfidf_embeddings_sample3000.json',   'tf_idf_embedding'),
}

frames = {}
for key, (path, col) in FILES.items():
    df = pd.read_json(path).dropna(subset=['query']).reset_index(drop=True)
    df['query'] = df['query'].astype(str)
    frames[key] = df
    print(f'{key}: n={len(df)} cols={list(df.columns)} emb_dim={len(df[col].iloc[0])}')

crag_openai: n=2706 cols=['domain', 'query', 'open_ai_embeddings', 'class'] emb_dim=1536
crag_tfidf: n=2706 cols=['domain', 'query', 'tf_idf_embedding'] emb_dim=6032
chat_openai: n=3000 cols=['query', 'tf_idf', 'open_ai_embeddings'] emb_dim=1536
chat_tfidf: n=3000 cols=['query', 'tf_idf', 'tf_idf_embedding'] emb_dim=6032


## 1. Train classifier on CRAG, predict chatbot_arena labels

Multiclass `LogisticRegression`, balanced class weights, 5-fold CV on CRAG for an honest accuracy estimate before predicting on chat.

In [23]:
def stack(df, col):
    return np.vstack(df[col].map(np.asarray).values).astype(np.float32)


def train_and_predict(crag_df, chat_df, emb_col, label):
    X_train = stack(crag_df, emb_col)
    y_train = crag_df['domain'].astype(str).str.lower().to_numpy()
    X_chat  = stack(chat_df, emb_col)

    clf = LogisticRegression(max_iter=2000, class_weight='balanced')
    cv = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy', n_jobs=1)
    print(f'[{label}] 5-fold CV accuracy on CRAG: {cv.mean():.3f} ± {cv.std():.3f}')

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_chat)
    print(f'[{label}] chatbot_arena predicted label counts:')
    print(pd.Series(y_pred).value_counts())
    return y_pred


y_pred_openai = train_and_predict(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', 'openai')
print()
y_pred_tfidf  = train_and_predict(frames['crag_tfidf'],  frames['chat_tfidf'],  'tf_idf_embedding',  'tfidf')

frames['chat_openai']['category'] = y_pred_openai
frames['chat_tfidf']['category']  = y_pred_tfidf
frames['crag_openai']['category'] = frames['crag_openai']['domain'].astype(str).str.lower()
frames['crag_tfidf']['category']  = frames['crag_tfidf']['domain'].astype(str).str.lower()

[openai] 5-fold CV accuracy on CRAG: 0.933 ± 0.003
[openai] chatbot_arena predicted label counts:
open       2665
finance     156
movie        63
sports       63
music        53
Name: count, dtype: int64

[tfidf] 5-fold CV accuracy on CRAG: 0.891 ± 0.015
[tfidf] chatbot_arena predicted label counts:
open       2557
movie       214
sports       82
finance      78
music        69
Name: count, dtype: int64


## 2. UMAP 3D — unsupervised vs supervised

Helper builds the combined matrix (CRAG + chat), runs UMAP (optionally supervised with `target_weight`), drops top 1% outliers from the median, re-centers, returns a DataFrame ready to plot.

In [13]:
# All helpers + plotting constants in one cell. Run this BEFORE any plot cell.
DROP_PCT = 1.0
ALL_CATS = ['finance', 'movie', 'music', 'sports', 'open']
palette = px.colors.qualitative.Bold
COLOR_MAP = {c: palette[i % len(palette)] for i, c in enumerate(ALL_CATS)}
SYMBOL = {'chatbot_arena': 'circle', 'crag': 'cross'}  # cross == '+'


def stack(df, col):
    return np.vstack(df[col].map(np.asarray).values).astype(np.float32)


def run_umap(crag_df, chat_df, emb_col, target_weight=None, y_col='category', label=''):
    crag_df = crag_df.copy(); crag_df['source'] = 'crag';          crag_df['origin'] = 0
    chat_df = chat_df.copy(); chat_df['source'] = 'chatbot_arena'; chat_df['origin'] = 1
    combined = pd.concat([crag_df, chat_df], ignore_index=True)
    X = stack(combined, emb_col)

    kwargs = dict(n_components=3, n_neighbors=20, min_dist=0.1, metric='cosine', random_state=42)
    if target_weight is not None:
        kwargs['target_weight'] = target_weight
        kwargs['target_metric'] = 'categorical'
    reducer = umap.UMAP(**kwargs)

    if target_weight is not None:
        y = LabelEncoder().fit_transform(combined[y_col].astype(str).values)
        emb3d = reducer.fit_transform(X, y=y)
    else:
        emb3d = reducer.fit_transform(X)

    center = np.median(emb3d, axis=0)
    dist = np.linalg.norm(emb3d - center, axis=1)
    cutoff = np.percentile(dist, 100 - DROP_PCT)
    keep = dist <= cutoff
    print(f'[{label}] dropped {(~keep).sum()} outliers, kept {keep.sum()}')

    out = combined.loc[keep, ['query', 'category', 'source']].reset_index(drop=True).copy()
    emb3d = emb3d[keep]
    center = np.median(emb3d, axis=0)
    out['umap_x'] = emb3d[:, 0] - center[0]
    out['umap_y'] = emb3d[:, 1] - center[1]
    out['umap_z'] = emb3d[:, 2] - center[2]
    return out


def plot_umap_3d(df, title):
    fig = go.Figure()
    for source, sym in SYMBOL.items():
        for cat in ALL_CATS:
            sub = df[(df['source'] == source) & (df['category'] == cat)]
            if len(sub) == 0:
                continue
            fig.add_trace(go.Scatter3d(
                x=sub['umap_x'], y=sub['umap_y'], z=sub['umap_z'],
                mode='markers',
                name=f'{source} | {cat}',
                marker=dict(
                    size=3 if source == 'chatbot_arena' else 5,
                    symbol=sym,
                    color=COLOR_MAP[cat],
                    opacity=0.75,
                    line=dict(width=0),
                ),
                hovertext=sub['query'].str.slice(0, 120),
                hoverinfo='text+name',
            ))
    axis_max = float(np.abs(df[['umap_x', 'umap_y', 'umap_z']].values).max()) * 1.05
    r = [-axis_max, axis_max]
    fig.update_layout(
        title=title,
        width=1100, height=800,
        scene=dict(
            xaxis=dict(title='UMAP-1', range=r, zeroline=True),
            yaxis=dict(title='UMAP-2', range=r, zeroline=True),
            zaxis=dict(title='UMAP-3', range=r, zeroline=True),
            aspectmode='cube',
        ),
        legend=dict(itemsizing='constant', groupclick='toggleitem'),
    )
    return fig

print('helpers ready: stack, run_umap, plot_umap_3d')

helpers ready: stack, run_umap, plot_umap_3d


### 2a. OpenAI — unsupervised

In [14]:
df_openai_unsup = run_umap(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', target_weight=None, label='openai unsup')
fig_openai_unsup = plot_umap_3d(df_openai_unsup, 'UMAP 3D — OpenAI, unsupervised (dot=chatbot_arena, +=crag)')
fig_openai_unsup.show()

/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[openai unsup] dropped 58 outliers, kept 5648


### 2b. OpenAI — supervised by class, target_weight=0.001

In [15]:
df_openai_sup = run_umap(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', target_weight=0.001, y_col='category', label='openai sup class')
fig_openai_sup = plot_umap_3d(df_openai_sup, 'UMAP 3D — OpenAI, supervised by class (target_weight=0.001)')
fig_openai_sup.show()

/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[openai sup class] dropped 58 outliers, kept 5648


### 2c. OpenAI — supervised by origin (crag=0, chat=1), target_weight=0.001

In [16]:
df_openai_sup_origin = run_umap(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', target_weight=0.001, y_col='origin', label='openai sup origin')
fig_openai_sup_origin = plot_umap_3d(df_openai_sup_origin, 'UMAP 3D — OpenAI, supervised by origin (target_weight=0.001)')
fig_openai_sup_origin.show()

/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[openai sup origin] dropped 58 outliers, kept 5648


### 2d. TF-IDF — unsupervised

In [17]:
df_tfidf_unsup = run_umap(frames['crag_tfidf'], frames['chat_tfidf'], 'tf_idf_embedding', target_weight=None, label='tfidf unsup')
fig_tfidf_unsup = plot_umap_3d(df_tfidf_unsup, 'UMAP 3D — TF-IDF, unsupervised (dot=chatbot_arena, +=crag)')
fig_tfidf_unsup.show()

/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[tfidf unsup] dropped 58 outliers, kept 5648


### 2e. TF-IDF — supervised by class, target_weight=0.001

In [18]:
df_tfidf_sup = run_umap(frames['crag_tfidf'], frames['chat_tfidf'], 'tf_idf_embedding', target_weight=0.001, y_col='category', label='tfidf sup class')
fig_tfidf_sup = plot_umap_3d(df_tfidf_sup, 'UMAP 3D — TF-IDF, supervised by class (target_weight=0.001)')
fig_tfidf_sup.show()

/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[tfidf sup class] dropped 58 outliers, kept 5648


### 2f. TF-IDF — supervised by origin (crag=0, chat=1), target_weight=0.001

In [19]:
df_tfidf_sup_origin = run_umap(frames['crag_tfidf'], frames['chat_tfidf'], 'tf_idf_embedding', target_weight=0.001, y_col='origin', label='tfidf sup origin')
fig_tfidf_sup_origin = plot_umap_3d(df_tfidf_sup_origin, 'UMAP 3D — TF-IDF, supervised by origin (target_weight=0.001)')
fig_tfidf_sup_origin.show()

/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[tfidf sup origin] dropped 58 outliers, kept 5648


## 3. Save HTML (optional)

In [20]:
for name, fig in [
    ('umap_3d_openai_unsup.html',           fig_openai_unsup),
    ('umap_3d_openai_sup_class_w0.001.html', fig_openai_sup),
    ('umap_3d_openai_sup_origin_w0.001.html', fig_openai_sup_origin),
    ('umap_3d_tfidf_unsup.html',            fig_tfidf_unsup),
    ('umap_3d_tfidf_sup_class_w0.001.html',  fig_tfidf_sup),
    ('umap_3d_tfidf_sup_origin_w0.001.html', fig_tfidf_sup_origin),
]:
    out = DATA_DIR / name
    fig.write_html(str(out))
    print('saved ->', out)

saved -> /Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed/umap_3d_openai_unsup.html
saved -> /Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed/umap_3d_openai_sup_class_w0.001.html
saved -> /Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed/umap_3d_openai_sup_origin_w0.001.html
saved -> /Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed/umap_3d_tfidf_unsup.html
saved -> /Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed/umap_3d_tfidf_sup_class_w0.001.html
saved -> /Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed/umap_3d_tfidf_sup_origin_w0.001.html
